In [ ]:
import os
import joblib
import warnings
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import (
    confusion_matrix, accuracy_score, f1_score, roc_auc_score
)
warnings.filterwarnings('ignore')
np.random.seed(42)

data_path = 'feature_matrix_3class_with_split.csv'   
model_save_path = 'rf_model_final.pkl'

feat_cols = [
    'force_vas_lat_r_delta', 'q_knee_angle_r_std',
    'force_vas_med_r_delta', 'force_vas_lat_l_delta',
    'force_vas_med_l_std', 'force_vas_lat_l_std',
    'force_vas_med_l_delta', 'q_knee_angle_r_range'
]

best_params = {
    'n_estimators': 20,
    'max_depth': 3,
    'min_samples_split': 40,
    'min_samples_leaf': 20,
    'max_features': 4,
    'max_leaf_nodes': 5,
    'ccp_alpha': 0.01,
    'class_weight': 'balanced',
    'random_state': 42,
    'n_jobs': -1
}

df = pd.read_csv(data_path)
train_df = df[df['set'] == 'train'].copy()
test_df  = df[df['set'] == 'test'].copy()

X_train = train_df[feat_cols].values
y_train = train_df['label'].values
X_test  = test_df[feat_cols].values
y_test  = test_df['label'].values

print(f"[INFO] Training samples: {X_train.shape[0]}")
print(f"[INFO] Test samples:     {X_test.shape[0]}")
print(f"[INFO] Training class distribution: {np.bincount(y_train)}")
print(f"[INFO] Test class distribution:     {np.bincount(y_test)}")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

rf = RandomForestClassifier(**best_params)
rf.fit(X_train_scaled, y_train)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
y_train_cv_pred = cross_val_predict(rf, X_train_scaled, y_train, cv=cv, method='predict')
y_train_cv_prob = cross_val_predict(rf, X_train_scaled, y_train, cv=cv, method='predict_proba')

y_test_pred = rf.predict(X_test_scaled)
y_test_prob = rf.predict_proba(X_test_scaled)

def evaluate(y_true, y_pred, y_prob, title):
    cm = confusion_matrix(y_true, y_pred)
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='macro')
    n_classes = 3
    y_bin = label_binarize(y_true, classes=range(n_classes))
    auc = roc_auc_score(y_bin, y_prob, average='macro', multi_class='ovr')
    
    print(f"\n{'='*50}")
    print(f" {title}")
    print('='*50)
    print(f"Confusion Matrix:\n{cm}")
    print(f"\nMacro AUC: {auc:.4f}")
    print(f"Macro F1 : {f1:.4f}")
    print(f"Accuracy : {acc:.4f}")
    return cm, auc, f1, acc
    
evaluate(y_train, y_train_cv_pred, y_train_cv_prob, "TRAINING (5-fold CV)")
evaluate(y_test,  y_test_pred,      y_test_prob,      "TEST (unseen)")

model_package = {
    'model': rf,
    'scaler': scaler,
    'selected_features': feat_cols,
    'best_params': best_params
}
joblib.dump(model_package, model_save_path)
print(f"\n[SAVED] Model + scaler → {model_save_path}")

In [ ]:
import os
import joblib
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix
warnings.filterwarnings('ignore')
np.random.seed(42)

plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.size'] = 10
plt.rcParams['axes.unicode_minus'] = False

print("="*80)
print("SVM Three-Class Model (Strong Regularization)")
print("="*80)

data_path = 'feature_matrix_3class_with_split.csv'
model_save_path = 'svm_model_final.pkl'

feat_cols = [
    'force_vas_lat_r_delta', 'q_knee_angle_r_std',
    'force_vas_med_r_delta', 'force_vas_lat_l_delta',
    'force_vas_med_l_std', 'force_vas_lat_l_std',
    'force_vas_med_l_delta', 'q_knee_angle_r_range'
]
class_names = ['Low Cost', 'Medium Cost', 'High Cost']

df = pd.read_csv(data_path)
train_df = df[df['set'] == 'train'].copy()
test_df = df[df['set'] == 'test'].copy()

X_train = train_df[feat_cols].values
y_train = train_df['label'].values
X_test  = test_df[feat_cols].values
y_test  = test_df['label'].values

print(f"Training samples: {X_train.shape[0]}")
print(f"Test samples: {X_test.shape[0]}")
for i, name in enumerate(class_names):
    print(f"  {name}: train={np.sum(y_train==i)}, test={np.sum(y_test==i)}")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

def calculate_metrics(y_true, y_pred, y_prob):
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='macro')
    cm = confusion_matrix(y_true, y_pred)
    y_bin = label_binarize(y_true, classes=[0,1,2])
    auc = roc_auc_score(y_bin, y_prob, average='macro', multi_class='ovr') if y_prob is not None else 0.0
    return cm, auc, f1, acc

param_grid = {
    'C': [0.01, 0.05, 0.1, 0.5, 1.0],
    'kernel': ['rbf', 'linear'],
    'gamma': ['scale', 0.01, 0.05, 0.1],
    'class_weight': ['balanced'],
    'decision_function_shape': ['ovr']
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
svm = SVC(probability=True, random_state=42, cache_size=1000)
grid = GridSearchCV(svm, param_grid, cv=cv, scoring='f1_macro', n_jobs=-1, verbose=0)
grid.fit(X_train_scaled, y_train)

best_svm = grid.best_estimator_
print(f"\nBest parameters: {grid.best_params_}")
cv_train_score = grid.cv_results_['mean_train_score'][grid.best_index_]
cv_val_score = grid.cv_results_['mean_test_score'][grid.best_index_]
print(f"CV train F1: {cv_train_score:.4f}, CV val F1: {cv_val_score:.4f}")

y_train_cv_pred = cross_val_predict(best_svm, X_train_scaled, y_train, cv=cv, method='predict')
y_train_cv_prob = cross_val_predict(best_svm, X_train_scaled, y_train, cv=cv, method='predict_proba')
y_test_pred = best_svm.predict(X_test_scaled)
y_test_prob = best_svm.predict_proba(X_test_scaled)

print("\n===== TRAINING (5-fold CV) =====")
cm_cv, auc_cv, f1_cv, acc_cv = calculate_metrics(y_train, y_train_cv_pred, y_train_cv_prob)
print(f"Confusion Matrix:\n{cm_cv}")
print(f"AUC={auc_cv:.4f}, F1={f1_cv:.4f}, ACC={acc_cv:.4f}")

print("\n===== TEST (unseen) =====")
cm_test, auc_test, f1_test, acc_test = calculate_metrics(y_test, y_test_pred, y_test_prob)
print(f"Confusion Matrix:\n{cm_test}")
print(f"AUC={auc_test:.4f}, F1={f1_test:.4f}, ACC={acc_test:.4f}")

model_pkg = {
    'model': best_svm,
    'scaler': scaler,
    'selected_features': feat_cols,
    'best_params': grid.best_params_
}
joblib.dump(model_pkg, model_save_path)
print(f"\nModel saved to {model_save_path}")

In [ ]:
import os
import joblib
import warnings
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix
warnings.filterwarnings('ignore')
np.random.seed(42)

data_path = 'feature_matrix_3class_with_split.csv'
feat_cols = [
    'force_vas_lat_r_delta', 'q_knee_angle_r_std',
    'force_vas_med_r_delta', 'force_vas_lat_l_delta',
    'force_vas_med_l_std', 'force_vas_lat_l_std',
    'force_vas_med_l_delta', 'q_knee_angle_r_range'
]
class_names = ['Low Cost', 'Medium Cost', 'High Cost']

df = pd.read_csv(data_path)
train_df = df[df['set'] == 'train']
test_df  = df[df['set'] == 'test']

X_train = train_df[feat_cols].values
y_train = train_df['label'].values
X_test  = test_df[feat_cols].values
y_test  = test_df['label'].values

print(f"Train: {X_train.shape[0]}, Test: {X_test.shape[0]}")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

def calc_metrics(y_true, y_pred, y_prob):
    acc = accuracy_score(y_true, y_pred)
    f1  = f1_score(y_true, y_pred, average='macro')
    cm  = confusion_matrix(y_true, y_pred)
    y_bin = label_binarize(y_true, classes=[0,1,2])
    auc = roc_auc_score(y_bin, y_prob, average='macro', multi_class='ovr') if y_prob is not None else 0.0
    return cm, auc, f1, acc

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# ===== XGBoost =====
print("\n===== XGBoost =====")
xgb_params = {
    'n_estimators': [30,50,80],
    'max_depth': [2,3,4],
    'learning_rate': [0.01,0.05,0.1],
    'min_child_weight': [5,10,20],
    'subsample': [0.6,0.7,0.8],
    'colsample_bytree': [0.5,0.6,0.7],
    'reg_alpha': [0.1,0.5,1.0],
    'reg_lambda': [1.0,2.0,5.0],
    'gamma': [0.1,0.5,1.0]
}
xgb_model = xgb.XGBClassifier(objective='multi:softproba', num_class=3,
                               random_state=42, n_jobs=-1, verbosity=0)
xgb_grid = GridSearchCV(xgb_model, xgb_params, cv=cv, scoring='f1_macro', n_jobs=-1)
xgb_grid.fit(X_train, y_train)
best_xgb = xgb_grid.best_estimator_
print(f"Best params: {xgb_grid.best_params_}")

y_tr_cv = cross_val_predict(best_xgb, X_train, y_train, cv=cv, method='predict')
y_tr_cv_prob = cross_val_predict(best_xgb, X_train, y_train, cv=cv, method='predict_proba')
y_te = best_xgb.predict(X_test)
y_te_prob = best_xgb.predict_proba(X_test)

cm_tr, auc_tr, f1_tr, acc_tr = calc_metrics(y_train, y_tr_cv, y_tr_cv_prob)
cm_te, auc_te, f1_te, acc_te = calc_metrics(y_test, y_te, y_te_prob)
print(f"Train (CV): AUC={auc_tr:.4f}, F1={f1_tr:.4f}, ACC={acc_tr:.4f}")
print(f"Test      : AUC={auc_te:.4f}, F1={f1_te:.4f}, ACC={acc_te:.4f}")
print(f"Confusion Matrix (Test):\n{cm_te}")

joblib.dump({'model':best_xgb,'scaler':None,'selected_features':feat_cols,
             'best_params':xgb_grid.best_params_}, 'xgb_model_final.pkl')

# ===== Logistic Regression =====
print("\n===== Logistic Regression =====")
lr_params = {
    'C': [0.001,0.01,0.1,0.5,1.0],
    'penalty': ['l1','l2'],
    'solver': ['saga'],
    'max_iter': [2000],
    'class_weight': ['balanced'],
    'multi_class': ['multinomial']
}
lr_model = LogisticRegression(random_state=42)
lr_grid = GridSearchCV(lr_model, lr_params, cv=cv, scoring='f1_macro', n_jobs=-1)
lr_grid.fit(X_train_scaled, y_train)
best_lr = lr_grid.best_estimator_
print(f"Best params: {lr_grid.best_params_}")

y_tr_cv = cross_val_predict(best_lr, X_train_scaled, y_train, cv=cv, method='predict')
y_tr_cv_prob = cross_val_predict(best_lr, X_train_scaled, y_train, cv=cv, method='predict_proba')
y_te = best_lr.predict(X_test_scaled)
y_te_prob = best_lr.predict_proba(X_test_scaled)

cm_tr, auc_tr, f1_tr, acc_tr = calc_metrics(y_train, y_tr_cv, y_tr_cv_prob)
cm_te, auc_te, f1_te, acc_te = calc_metrics(y_test, y_te, y_te_prob)
print(f"Train (CV): AUC={auc_tr:.4f}, F1={f1_tr:.4f}, ACC={acc_tr:.4f}")
print(f"Test      : AUC={auc_te:.4f}, F1={f1_te:.4f}, ACC={acc_te:.4f}")
print(f"Confusion Matrix (Test):\n{cm_te}")

joblib.dump({'model':best_lr,'scaler':scaler,'selected_features':feat_cols,
             'best_params':lr_grid.best_params_}, 'lr_model_final.pkl')

print("\nModels saved: xgb_model_final.pkl, lr_model_final.pkl")

In [ ]:
import os
import joblib
import warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix

warnings.filterwarnings('ignore')
np.random.seed(42)

data_path = 'feature_matrix_3class_with_split.csv'
feat_cols = [
    'force_vas_lat_r_delta', 'q_knee_angle_r_std',
    'force_vas_med_r_delta', 'force_vas_lat_l_delta',
    'force_vas_med_l_std', 'force_vas_lat_l_std',
    'force_vas_med_l_delta', 'q_knee_angle_r_range'
]
class_names = ['Low Cost', 'Medium Cost', 'High Cost']

df = pd.read_csv(data_path)
train_df = df[df['set'] == 'train']
test_df  = df[df['set'] == 'test']

X_train = train_df[feat_cols].values
y_train = train_df['label'].values
X_test  = test_df[feat_cols].values
y_test  = test_df['label'].values

print(f"Train: {X_train.shape[0]}, Test: {X_test.shape[0]}")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

def calc_metrics(y_true, y_pred, y_prob):
    acc = accuracy_score(y_true, y_pred)
    f1  = f1_score(y_true, y_pred, average='macro')
    cm  = confusion_matrix(y_true, y_pred)
    y_bin = label_binarize(y_true, classes=[0,1,2])
    auc = roc_auc_score(y_bin, y_prob, average='macro', multi_class='ovr') if y_prob is not None else 0.0
    return cm, auc, f1, acc

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# ===== GBC =====
print("\n===== Gradient Boosting =====")
gbc_params = {
    'n_estimators': [30,50,80],
    'max_depth': [2,3,4],
    'learning_rate': [0.01,0.05,0.1],
    'min_samples_split': [20,30,50],
    'min_samples_leaf': [10,15,20],
    'subsample': [0.6,0.7,0.8],
    'max_features': ['sqrt',0.5]
}
gbc = GradientBoostingClassifier(random_state=42)
gbc_grid = GridSearchCV(gbc, gbc_params, cv=cv, scoring='f1_macro', n_jobs=-1)
gbc_grid.fit(X_train, y_train)
best_gbc = gbc_grid.best_estimator_
print(f"Best params: {gbc_grid.best_params_}")

y_tr_cv = cross_val_predict(best_gbc, X_train, y_train, cv=cv, method='predict')
y_tr_cv_prob = cross_val_predict(best_gbc, X_train, y_train, cv=cv, method='predict_proba')
y_te = best_gbc.predict(X_test)
y_te_prob = best_gbc.predict_proba(X_test)
cm_tr, auc_tr, f1_tr, acc_tr = calc_metrics(y_train, y_tr_cv, y_tr_cv_prob)
cm_te, auc_te, f1_te, acc_te = calc_metrics(y_test, y_te, y_te_prob)
print(f"Train (CV): AUC={auc_tr:.4f}, F1={f1_tr:.4f}, ACC={acc_tr:.4f}")
print(f"Test      : AUC={auc_te:.4f}, F1={f1_te:.4f}, ACC={acc_te:.4f}")
print(f"CM (Test):\n{cm_te}")

joblib.dump({'model':best_gbc,'scaler':None,'selected_features':feat_cols,
             'best_params':gbc_grid.best_params_}, 'gbc_model_final.pkl')

# ===== KNN =====
print("\n===== KNN =====")
knn_params = {
    'n_neighbors': [5,7,9,11,15,21],
    'weights': ['uniform','distance'],
    'metric': ['euclidean','manhattan','minkowski'],
    'p': [1,2]
}
knn = KNeighborsClassifier(n_jobs=-1)
knn_grid = GridSearchCV(knn, knn_params, cv=cv, scoring='f1_macro', n_jobs=-1)
knn_grid.fit(X_train_scaled, y_train)
best_knn = knn_grid.best_estimator_
print(f"Best params: {knn_grid.best_params_}")

y_tr_cv = cross_val_predict(best_knn, X_train_scaled, y_train, cv=cv, method='predict')
y_tr_cv_prob = cross_val_predict(best_knn, X_train_scaled, y_train, cv=cv, method='predict_proba')
y_te = best_knn.predict(X_test_scaled)
y_te_prob = best_knn.predict_proba(X_test_scaled)
cm_tr, auc_tr, f1_tr, acc_tr = calc_metrics(y_train, y_tr_cv, y_tr_cv_prob)
cm_te, auc_te, f1_te, acc_te = calc_metrics(y_test, y_te, y_te_prob)
print(f"Train (CV): AUC={auc_tr:.4f}, F1={f1_tr:.4f}, ACC={acc_tr:.4f}")
print(f"Test      : AUC={auc_te:.4f}, F1={f1_te:.4f}, ACC={acc_te:.4f}")
print(f"CM (Test):\n{cm_te}")

joblib.dump({'model':best_knn,'scaler':scaler,'selected_features':feat_cols,
             'best_params':knn_grid.best_params_}, 'knn_model_final.pkl')

# ===== MLP =====
print("\n===== MLP =====")
mlp_params = {
    'hidden_layer_sizes': [(16,),(32,),(16,8),(32,16)],
    'activation': ['relu','tanh'],
    'alpha': [0.1,0.5,1.0,2.0],
    'learning_rate': ['constant','adaptive'],
    'learning_rate_init': [0.001,0.01],
    'early_stopping': [True],
    'validation_fraction': [0.15],
    'n_iter_no_change': [10]
}
mlp = MLPClassifier(max_iter=500, random_state=42, solver='adam')
mlp_grid = GridSearchCV(mlp, mlp_params, cv=cv, scoring='f1_macro', n_jobs=-1)
mlp_grid.fit(X_train_scaled, y_train)
best_mlp = mlp_grid.best_estimator_
print(f"Best params: {mlp_grid.best_params_}")

y_tr_cv = cross_val_predict(best_mlp, X_train_scaled, y_train, cv=cv, method='predict')
y_tr_cv_prob = cross_val_predict(best_mlp, X_train_scaled, y_train, cv=cv, method='predict_proba')
y_te = best_mlp.predict(X_test_scaled)
y_te_prob = best_mlp.predict_proba(X_test_scaled)
cm_tr, auc_tr, f1_tr, acc_tr = calc_metrics(y_train, y_tr_cv, y_tr_cv_prob)
cm_te, auc_te, f1_te, acc_te = calc_metrics(y_test, y_te, y_te_prob)
print(f"Train (CV): AUC={auc_tr:.4f}, F1={f1_tr:.4f}, ACC={acc_tr:.4f}")
print(f"Test      : AUC={auc_te:.4f}, F1={f1_te:.4f}, ACC={acc_te:.4f}")
print(f"CM (Test):\n{cm_te}")

joblib.dump({'model':best_mlp,'scaler':scaler,'selected_features':feat_cols,
             'best_params':mlp_grid.best_params_}, 'mlp_model_final.pkl')

print("\nModels saved: gbc_model_final.pkl, knn_model_final.pkl, mlp_model_final.pkl")

In [ ]:
import os
import joblib
import warnings
import numpy as np
import pandas as pd
import json
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix
warnings.filterwarnings('ignore')
np.random.seed(42)

base_folder = '.'
feature_folder = os.path.join(base_folder, 'Results_Feature_Selection_3Class')
model_paths = {
    'RandomForest': os.path.join(base_folder, 'Results_RF_3Class_StrongReg', 'RF_3class_strong_reg_model.pkl'),
    'SVM': os.path.join(base_folder, 'Results_SVM_3Class', 'SVM_3class_model.pkl'),
    'XGBoost': os.path.join(base_folder, 'Results_XGBoost_3Class', 'XGBoost_3class_model.pkl'),
    'LogisticRegression': os.path.join(base_folder, 'Results_LR_3Class', 'LR_3class_model.pkl'),
    'GradientBoosting': os.path.join(base_folder, 'Results_GBC_3Class', 'GBC_3class_model.pkl'),
    'KNN': os.path.join(base_folder, 'Results_KNN_3Class', 'KNN_3class_model.pkl'),
    'MLP': os.path.join(base_folder, 'Results_MLP_3Class', 'MLP_3class_model.pkl')
}

class_names = ['Low Cost', 'Medium Cost', 'High Cost']
output_folder = os.path.join(base_folder, 'Results_Ensemble_3Class')
os.makedirs(output_folder, exist_ok=True)

data_path = os.path.join(feature_folder, 'feature_matrix_3class_with_split.csv')
df = pd.read_csv(data_path)
train_df = df[df['set'] == 'train']
test_df  = df[df['set'] == 'test']

feat_cols = [
    'force_vas_lat_r_delta', 'q_knee_angle_r_std',
    'force_vas_med_r_delta', 'force_vas_lat_l_delta',
    'force_vas_med_l_std', 'force_vas_lat_l_std',
    'force_vas_med_l_delta', 'q_knee_angle_r_range'
]

X_train = train_df[feat_cols].values
y_train = train_df['label'].values
X_test  = test_df[feat_cols].values
y_test  = test_df['label'].values

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

models = {}
scalers_dict = {}
for name, path in model_paths.items():
    if not os.path.exists(path):
        print(f"Missing {name} at {path}")
        continue
    pkg = joblib.load(path)
    models[name] = pkg['model']
    scalers_dict[name] = pkg.get('scaler', None)
print(f"Loaded {len(models)} models")

def model_predict(name, model, X, X_scaled):
    if name in ['KNN','MLP','SVM','LogisticRegression']:
        return model.predict(X_scaled), model.predict_proba(X_scaled)
    return model.predict(X), model.predict_proba(X)

results = {m: [] for m in ['AUC','F1','ACC','PRE','SEN','SPE']}
preds = {}
for name, model in models.items():
    y_pred, y_prob = model_predict(name, model, X_test, X_test_scaled)
    preds[name] = (y_pred, y_prob)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='macro')
    y_bin = label_binarize(y_test, classes=[0,1,2])
    auc = roc_auc_score(y_bin, y_prob, average='macro', multi_class='ovr')
    cm = confusion_matrix(y_test, y_pred)
    n_classes = 3
    sens, spec = [], []
    for i in range(n_classes):
        tp = cm[i,i]; fn = cm[i,:].sum()-tp; fp = cm[:,i].sum()-tp; tn = cm.sum()-tp-fn-fp
        sens.append(tp/(tp+fn) if (tp+fn)>0 else 0)
        spec.append(tn/(tn+fp) if (tn+fp)>0 else 0)
    results['AUC'].append(auc)
    results['F1'].append(f1)
    results['ACC'].append(acc)
    results['PRE'].append(accuracy_score(y_test, y_pred))
    results['SEN'].append(np.mean(sens))
    results['SPE'].append(np.mean(spec))
    print(f"{name}: AUC={auc:.4f}, F1={f1:.4f}, ACC={acc:.4f}")

results_df = pd.DataFrame(results, index=models.keys())

def topsis(data, weight=None):
    norm = data / np.sqrt((data**2).sum(axis=0))
    if weight is None:
        weight = np.ones(len(data.columns))/len(data.columns)
    weighted = norm * weight
    pis = weighted.max(axis=0)
    nis = weighted.min(axis=0)
    d_pos = np.sqrt(((weighted-pis)**2).sum(axis=1))
    d_neg = np.sqrt(((weighted-nis)**2).sum(axis=1))
    scores = d_neg/(d_pos+d_neg+1e-9)
    weights = scores/scores.sum()
    return weights

weights = topsis(results_df)
print("\nTOPSIS weights:")
for name, w in zip(models.keys(), weights):
    print(f"  {name}: {w:.4f}")

class EnsembleModel(BaseEstimator, ClassifierMixin):
    def __init__(self, models, weights, n_classes=3):
        self.models = models
        self.weights = np.array(weights)
        self.n_classes = n_classes
        self.scaler = StandardScaler()
        self.classes_ = np.array([0,1,2])
    def fit(self, X, y):
        self.scaler.fit(X)
        return self
    def predict_proba(self, X):
        probas = np.zeros((X.shape[0], self.n_classes))
        X_scaled = self.scaler.transform(X)
        for i, (name, model) in enumerate(self.models.items()):
            if name in ['KNN','MLP','SVM','LogisticRegression']:
                X_in = X_scaled
            else:
                X_in = X
            proba = model.predict_proba(X_in)
            probas += proba * self.weights[i]
        return probas
    def predict(self, X):
        return np.argmax(self.predict_proba(X), axis=1)

ens = EnsembleModel(models, weights)
ens.fit(X_train, y_train)

y_te_pred = ens.predict(X_test)
y_te_prob = ens.predict_proba(X_test)
cm_ens = confusion_matrix(y_test, y_te_pred)
acc_ens = accuracy_score(y_test, y_te_pred)
f1_ens = f1_score(y_test, y_te_pred, average='macro')
auc_ens = roc_auc_score(label_binarize(y_test, classes=[0,1,2]), y_te_prob, average='macro', multi_class='ovr')

print("\n===== Ensemble (TOPSIS) =====")
print(f"Confusion Matrix:\n{cm_ens}")
print(f"AUC={auc_ens:.4f}, F1={f1_ens:.4f}, ACC={acc_ens:.4f}")

joblib.dump({'model':ens, 'weights':weights, 'base_models':models,
             'test_metrics':{'AUC':auc_ens,'F1':f1_ens,'ACC':acc_ens},
             'confusion_matrix':cm_ens}, os.path.join(output_folder, 'Ensemble_model.pkl'))
print(f"Model saved to {output_folder}/Ensemble_model.pkl")